# Project 3, Part 3, Create graph 1 database in Neo4j, Year 2015
## Graph 1: one-way relationships from coo (country of origin) to coa (country of asylum)

## Included Modules and Packages

In [1]:
import neo4j

import csv

import math
import numpy as np
import pandas as pd

import psycopg2

## Supporting code

In [2]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [3]:
session = driver.session(database="neo4j")

In [4]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [5]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [6]:
def my_neo4j_number_nodes_relationships():
    "print the number of nodes and relationships"
   
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    print("-------------------------")
    print("  Nodes:", number_nodes)
    print("  Relationships:", number_relationships)
    print("-------------------------")

In [7]:
def my_neo4j_create_node(country_name):
    "create a node with label Country"
    
    query = """
    
    CREATE (:Country {name: $country_name})
    
    """
    
    session.run(query, country_name=country_name)

In [8]:
def my_neo4j_create_relationship_one_way(from_country, to_country, weight):
    "create a relationship one way between two countries with a weight"
    
    query = """
    
    MATCH (from:Country), 
          (to:Country)
    WHERE from.name = $from_country and to.name = $to_country
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_country=from_country, to_country=to_country, weight=weight)

In [9]:
def my_neo4j_create_relationship_two_way(from_country, to_country, weight):
    "create relationships two way between two countries with a weight"
    
    query = """
    
    MATCH (from:Country), 
          (to:Country)
    WHERE from.name = $from_country and to.name = $to_country
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_country=from_country, to_country=to_country, weight=weight)

In [10]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [11]:
cursor = connection.cursor()

## 3.3.1 Wipe out the Neo4j database

In [12]:
my_neo4j_wipe_out_database()

## 3.3.2 Verify the number of nodes and relationships

In [13]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 0
  Relationships: 0
-------------------------


## 3.3.3 Query list of countries and create the emigration and immigration nodes in the graph

In [14]:
connection.rollback()

query = """

(select coo as country
from stage_1_refugees)
union
(select coa as country
from stage_1_refugees)
order by country

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    country = row[0]
    
    my_neo4j_create_node(country)
    

## 3.3.4 Verify the number of nodes and relationships

In [15]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 212
  Relationships: 0
-------------------------


## 3.3.5 Query the list of all possible refugee movements and number of refugees for a particular year, create a relationship for each movement with number of refugees as the weight

- One way relationship: from `the country the refugees are emigrating from` to `the country the refugees are immigrating to`
- The weight of the relationship: the number of refugees
- Year: 2015

In [16]:
connection.rollback()

query = """

select coo as from_country,
       coa as to_country,
       refugees
from stage_1_refugees
where (coo <> coa) and (refugees::numeric > 0) and (year::numeric = 2015)
order by coo, coa, refugees

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    coo = row[0]
    coa = row[1]
    refugees = int(row[2])
    
    my_neo4j_create_relationship_one_way(coo, coa, refugees)

## 3.3.6 Verify the number of nodes and relationships

In [17]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 212
  Relationships: 3927
-------------------------


## 3.3.7 PageRank algorithm

**What are the most influential countries in granting asylum?**

In [18]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Country', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

In [19]:
query = """

CALL gds.pageRank.stream('ds_graph',
                         { maxIterations: $max_iterations,
                           dampingFactor: $damping_factor}
                         )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as page_rank
ORDER BY page_rank DESC, name ASC

"""

max_iterations = 20
damping_factor = 0.05

my_neo4j_run_query_pandas(query, max_iterations=max_iterations, damping_factor=damping_factor)

,name,page_rank
0,USA,2.272198
1,CAN,2.107804
2,GFR,1.530550
3,GBR,1.476312
4,AUL,1.438807
...,...,...
207,VAN,0.950000
208,VAT,0.950000
209,VCT,0.950000
210,WES,0.950000


## 3.3.8 Personalized PageRank

In [20]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Country', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

In [21]:
query = """

MATCH (siteA:Country {name: $source})
CALL gds.pageRank.stream('ds_graph', {
  maxIterations: $max_iterations,
  dampingFactor: $damping_factor,
  sourceNodes: [siteA]
})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as page_rank
ORDER BY score DESC, name ASC

"""

source = "USA"
max_iterations = 20
damping_factor = 0.85

my_neo4j_run_query_pandas(query, source=source, max_iterations=max_iterations, damping_factor=damping_factor)

,name,page_rank
0,USA,0.264876
1,GFR,0.129369
2,CAN,0.121543
3,SWE,0.095203
4,GBR,0.086805
...,...,...
207,VAN,0.000000
208,VAT,0.000000
209,VCT,0.000000
210,WES,0.000000


## 3.3.9 Harmonic centrality algorithm

In [22]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Country', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

In [23]:
query = """

CALL gds.closeness.harmonic.stream('ds_graph', {})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS country, score as closeness
ORDER BY closeness DESC

"""

my_neo4j_run_query_pandas(query)

,country,closeness
0,CAN,0.848341
1,USA,0.846761
2,GFR,0.763033
3,GBR,0.743286
4,SWE,0.715640
...,...,...
207,STA,0.000000
208,STK,0.000000
209,STP,0.000000
210,SUR,0.000000


## 3.3.10 Louvain modularity algorithm

In [24]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = """

CALL gds.graph.project('ds_graph', 'Country', 'LINK', 
                      {relationshipProperties: 'weight'})
"""

session.run(query)

In [25]:
query = """

CALL gds.louvain.stream('ds_graph', {includeIntermediateCommunities: true})
YIELD nodeId, communityId, intermediateCommunityIds
RETURN gds.util.asNode(nodeId).name AS country, communityId as community, intermediateCommunityIds as intermediate_community
ORDER BY community, country ASC
"""

my_neo4j_run_query_pandas(query)


,country,community,intermediate_community
0,TUV,6,"[6, 6, 6]"
1,VAN,14,"[14, 14, 14]"
2,VAT,15,"[15, 15, 15]"
3,WES,18,"[18, 18, 18]"
4,ABW,23,"[23, 23, 23]"
...,...,...,...
207,VEN,204,"[92, 165, 204]"
208,WSH,204,"[92, 165, 204]"
209,YEM,204,"[92, 165, 204]"
210,ZAM,204,"[92, 165, 204]"
